In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing, load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler

#  River
try:
    from river import linear_model, optim, tree, preprocessing as river_prep
    RIVER_AVAILABLE = True
    print("River: OK")
except ImportError:
    RIVER_AVAILABLE = False
    print("pip install river")

# CapyMOA 
try:
    from capymoa.stream import NumpyStream
    from capymoa.regressor import (
        SGDRegressor as CapySGD,
        PassiveAggressiveRegressor as CapyPA,
        StreamingGradientBoostedRegression as CapySGBR,
    )
    CAPYMOA_AVAILABLE = True
    print("CapyMOA: OK")
except ImportError:
    CAPYMOA_AVAILABLE = False
    print("pip install capymoa")

# PyTorch 
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    print("PyTorch: OK")
except ImportError:
    TORCH_AVAILABLE = False
    print("pip install torch")


def get_huber_env(w, x, y, delta=1.0):
    """Huber loss + gradient for Coin Betting """
    diff = float(np.dot(w, x)) - y
    if np.abs(diff) <= delta:
        return 0.5 * diff**2, diff * x
    return delta * (np.abs(diff) - 0.5 * delta), delta * np.sign(diff) * x


def huber_scalar(y_hat, y, delta=1.0):
    """Huber loss for River и CapyMOA."""
    diff = y_hat - y
    if np.abs(diff) <= delta:
        return 0.5 * diff**2
    return delta * (np.abs(diff) - 0.5 * delta)


def cumavg(losses):
    arr = np.array(losses, dtype=np.float64)
    if len(arr) == 0:
        return np.array([])
    for i in range(1, len(arr)):
        if not np.isfinite(arr[i]):
            arr[i] = arr[i - 1]
    arr[0] = arr[0] if np.isfinite(arr[0]) else 0.0
    return np.cumsum(arr) / np.arange(1, len(arr) + 1)

# COIN BETTING

class OGD:
    def __init__(self, dim, eta=0.01):
        self.w = np.zeros(dim)
        self.eta = eta

    def get_w(self): return self.w

    def update(self, g):
        self.w -= self.eta * g
        n = np.linalg.norm(self.w)
        if n > 50:
            self.w *= 50 / n


class COCOB_Standard:
    def __init__(self, dim, L_init=1.0):
        self.L      = np.full(dim, L_init)
        self.G      = np.full(dim, L_init)  
        self.theta  = np.zeros(dim)
        self.reward = np.zeros(dim)
        self.w      = np.zeros(dim)

    def get_w(self): return self.w

    def update(self, grad):
        g = -grad   
        self.G      += np.abs(g)
        self.reward += self.w * g
        self.theta  += g
        L    = np.where(self.L > 1e-12, self.L, 1e-12)
        x    = np.clip(2 * self.theta / (self.G + L), -50, 50)
        beta = (1 / L) * (2 / (1 + np.exp(-2 * x)) - 1)
        self.w = np.clip(beta * (L + self.reward), -1e5, 1e5)


class COCOB_Backprop:
    def __init__(self, dim, alpha=100.0):
        self.L      = np.zeros(dim)
        self.G      = np.zeros(dim)
        self.theta  = np.zeros(dim)
        self.reward = np.zeros(dim)
        self.w      = np.zeros(dim)
        self.alpha  = alpha

    def get_w(self): return self.w

    def update(self, grad):
        g = -grad
        self.L      = np.maximum(self.L, np.abs(g))
        self.G      += np.abs(g)
        self.reward = np.maximum(self.reward + self.w * g, 0)
        self.theta  += g
        L     = np.where(self.L > 1e-14, self.L, 1e-14)
        denom = L * np.maximum(self.G + L, self.alpha * L)
        self.w = (self.theta / denom) * (L + self.reward)


class ReductionToMagnitude:
    def __init__(self, dim):
        self.w_dir    = np.zeros(dim)
        self.L_z      = 1e-8
        self.G_z      = 1e-8
        self.theta_z  = 0.0
        self.reward_z = 0.0
        self.z        = 0.0
        self.t        = 0

    def get_w(self): return self.z * self.w_dir

    def update(self, grad):
        self.t += 1

        eta        = 1.0 / np.sqrt(self.t)
        self.w_dir -= eta * grad
        n          = np.linalg.norm(self.w_dir)
        if n > 1.0:
            self.w_dir /= n

        dn  = np.linalg.norm(self.w_dir)
        s   = np.dot(grad, self.w_dir / dn) if dn > 1e-12 else np.linalg.norm(grad)
        g_z = -s

        self.L_z      = max(self.L_z, abs(g_z))
        self.G_z      += abs(g_z)
        self.reward_z = max(self.reward_z + self.z * g_z, 0)
        self.theta_z  += g_z

        L         = max(self.L_z, 1e-14)
        denom     = L * max(self.G_z + L, 100 * L)
        self.z    = max((self.theta_z / denom) * (L + self.reward_z), 0)


# 3. PyTorch MLP

class OnlineMLP:
    def __init__(self, dim, hidden=(64, 32), lr=1e-3, delta=1.0):
        if not TORCH_AVAILABLE:
            raise RuntimeError("PyTorch not install")
        layers = []
        in_f   = dim
        for h in hidden:
            layers += [nn.Linear(in_f, h), nn.ReLU()]
            in_f = h
        layers.append(nn.Linear(in_f, 1))
        self.net = nn.Sequential(*layers)
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
        self.opt     = torch.optim.Adam(self.net.parameters(), lr=lr)
        self.loss_fn = nn.HuberLoss(delta=delta, reduction="mean")

    def predict_and_update(self, x_np, y_scalar):
        x_t   = torch.tensor(x_np,       dtype=torch.float32).unsqueeze(0)
        y_t   = torch.tensor([[y_scalar]], dtype=torch.float32)
        y_hat = self.net(x_t)
        loss  = self.loss_fn(y_hat, y_t)
        self.opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.net.parameters(), max_norm=1.0)
        self.opt.step()
        return float(loss.item()), float(y_hat.item())


def run_coinbetting(X, y):
    T, dim = X.shape
    opts = {
        "OGD (η=0.01)":           OGD(dim, 0.01),
        "Reduction to Magnitude":  ReductionToMagnitude(dim),
        "COCOB Standard":          COCOB_Standard(dim, 1.0),
        "COCOB Backprop":          COCOB_Backprop(dim, 100.0),
    }
    losses = {k: [] for k in opts}
    for t in range(T):
        for name, opt in opts.items():
            loss, grad = get_huber_env(opt.get_w(), X[t], y[t])
            losses[name].append(loss)
            opt.update(grad)
    return losses


def run_river(X, y):
    if not RIVER_AVAILABLE:
        return {}
    models = {
        "River: SGD LinearReg":  river_prep.StandardScaler() | linear_model.LinearRegression(
                                     optimizer=optim.SGD(lr=0.01),
                                     loss=optim.losses.Huber(epsilon=1.0)),
        "River: PA-Regressor":   river_prep.StandardScaler() | linear_model.PARegressor(
                                     C=0.01, mode=2, eps=0.1),
        "River: Hoeffding Tree": river_prep.StandardScaler() | tree.HoeffdingTreeRegressor(
                                     grace_period=50),
    }
    losses = {k: [] for k in models}
    for t in range(len(y)):
        xd = {i: float(X[t, i]) for i in range(X.shape[1])}
        yt = float(y[t])
        for name, model in models.items():
            yh = model.predict_one(xd) or 0.0
            losses[name].append(huber_scalar(yh, yt))
            model.learn_one(xd, yt)
    return losses


def run_capymoa(X, y):
    if not CAPYMOA_AVAILABLE:
        return {}

    ref_stream = NumpyStream(
        X.astype(np.float64), y.astype(np.float64), target_type="numeric"
    )
    schema = ref_stream.get_schema()

    model_factories = {
        "CapyMOA: SGD (Huber)":  lambda: CapySGD(schema=schema, loss="huber",
                                                   epsilon=1.0, eta0=0.01,
                                                   learning_rate="constant"),
        "CapyMOA: PA-Regressor": lambda: CapyPA(schema=schema),
        "CapyMOA: SGBR":         lambda: CapySGBR(schema=schema),
    }
    losses = {k: [] for k in model_factories}

    for name, factory in model_factories.items():
        stream = NumpyStream(
            X.astype(np.float64), y.astype(np.float64), target_type="numeric"
        )
        model = factory()
        for idx in range(len(y)):
            instance = stream.next_instance()
            if instance is None:
                break
            yh = model.predict(instance)
            yh = float(yh) if yh is not None else 0.0
            losses[name].append(huber_scalar(yh, float(y[idx])))
            model.train(instance)

    return losses


def run_neural(X, y, configs=None):

    if not TORCH_AVAILABLE:
        print("  PyTorch unavailable")
        return {}
    if configs is None:
        configs = [
            {"hidden": (64, 32),  "lr": 1e-3,  "label": "MLP (64→32, lr=1e-3)"},
            {"hidden": (128, 64), "lr": 5e-4,  "label": "MLP (128→64, lr=5e-4)"},
        ]
    dim    = X.shape[1]
    models = {cfg["label"]: OnlineMLP(dim, cfg["hidden"], cfg["lr"]) for cfg in configs}
    losses = {label: [] for label in models}
    for t in range(len(y)):
        for label, model in models.items():
            loss_val, _ = model.predict_and_update(X[t], float(y[t]))
            losses[label].append(loss_val)
    return losses


np.random.seed(42)

# California Housing
raw_ca   = fetch_california_housing()
X_ca_all = StandardScaler().fit_transform(raw_ca.data)
y_ca_all = StandardScaler().fit_transform(raw_ca.target.reshape(-1, 1)).ravel()
idx      = np.random.permutation(len(X_ca_all))
X_ca_all, y_ca_all = X_ca_all[idx], y_ca_all[idx]
X_ca, y_ca = X_ca_all, y_ca_all

rbf_ca = RBFSampler(gamma=0.1, n_components=200, random_state=42)
X_ca_f = rbf_ca.fit_transform(X_ca)

# Diabetes 
raw_db   = load_diabetes()
X_db_all = StandardScaler().fit_transform(raw_db.data)
y_db_all = StandardScaler().fit_transform(raw_db.target.reshape(-1, 1)).ravel()
idx2     = np.random.permutation(len(X_db_all))
X_db, y_db = X_db_all[idx2], y_db_all[idx2]

rbf_db = RBFSampler(gamma=0.5, n_components=200, random_state=42)
X_db_f = rbf_db.fit_transform(X_db)


print("\n" + "=" * 55)
print("CALIFORNIA HOUSING (T=5000)")
print("=" * 55)

print("[1/5] Coin Betting — линейные признаки...")
res_ca_lin = run_coinbetting(X_ca, y_ca)

print("[2/5] Coin Betting — Фурье φ(x)...")
res_ca_cb  = run_coinbetting(X_ca_f, y_ca)

print("[3/5] River — Фурье φ(x)...")
res_ca_rv  = run_river(X_ca_f, y_ca)

print("[4/5] CapyMOA — Фурье φ(x)...")
res_ca_cm  = run_capymoa(X_ca_f, y_ca)

print("[5/5] Neural Networks (MLP) — Фурье φ(x)...")
res_ca_nn  = run_neural(X_ca_f, y_ca)

print("\n" + "=" * 55)
print("DIABETES (T=442)")
print("=" * 55)

print("[1/3] Coin Betting — Фурье φ(x)...")
res_db_cb = run_coinbetting(X_db_f, y_db)

print("[2/3] River + CapyMOA — Фурье φ(x)...")
res_db_rv = run_river(X_db_f, y_db)
res_db_cm = run_capymoa(X_db_f, y_db)

print("[3/3] Neural Networks (MLP) — Фурье φ(x)...")
res_db_nn = run_neural(X_db_f, y_db)


print("\n" + "=" * 65)
print("ДИАГНОСТИКА: последние 200 мгновенных потерь (California, Фурье)")
print("=" * 65)
print(f"{'Алгоритм':<35} {'mean':>8} {'min':>8} {'max':>8} {'std':>8}")
print("-" * 65)
for name, losses in res_ca_cb.items():
    last   = np.array(losses[-200:], dtype=float)
    finite = last[np.isfinite(last)]
    if len(finite) == 0:
        print(f"{name:<35}   нет данных")
    else:
        print(f"{name:<35} {np.mean(finite):>8.4f} {np.min(finite):>8.4f} "
              f"{np.max(finite):>8.4f} {np.std(finite):>8.4f}")

print()
print("=" * 65)
print("ДИАГНОСТИКА: значения cumavg на конце прогона (California, Фурье)")
print("=" * 65)
print(f"{'Алгоритм':<35} {'t=4500':>8} {'t=4700':>8} {'t=4900':>8} {'t=5000':>8}")
print("-" * 65)
for name, losses in res_ca_cb.items():
    ca = cumavg(losses)
    if len(ca) < 5000:
        print(f"{name:<35}   короткий ряд ({len(ca)} итераций)")
        continue
    print(f"{name:<35} {ca[4499]:>8.4f} {ca[4699]:>8.4f} "
          f"{ca[4899]:>8.4f} {ca[4999]:>8.4f}")


COLORS = {
    "OGD (η=0.01)":                "#1f77b4",
    "Reduction to Magnitude":       "#ff7f0e",
    "COCOB Standard":               "#2ca02c",
    "COCOB Backprop":               "#d62728",

    "River: SGD LinearReg":         "#7bafd4",
    "River: PA-Regressor":          "#f4a460",
    "River: Hoeffding Tree":        "#90ee90",

    "CapyMOA: SGD (Huber)":         "#5a2d82",
    "CapyMOA: PA-Regressor":        "#b03060",
    "CapyMOA: SGBR":                "#8b4513",

    "MLP (64→32, lr=1e-3)":         "#e377c2",
    "MLP (128→64, lr=5e-4)":        "#bcbd22",
}

LS = {}
for n in ["OGD (η=0.01)", "Reduction to Magnitude", "COCOB Standard", "COCOB Backprop"]:
    LS[n] = "-"
for n in ["River: SGD LinearReg", "River: PA-Regressor", "River: Hoeffding Tree"]:
    LS[n] = "--"
for n in ["CapyMOA: SGD (Huber)", "CapyMOA: PA-Regressor", "CapyMOA: SGBR"]:
    LS[n] = "-."
for n in ["MLP (64→32, lr=1e-3)", "MLP (128→64, lr=5e-4)"]:
    LS[n] = (0, (3, 1, 1, 1))


def plot_group(ax, results_dict, title, ylim=(1e-2, 10)):
    for name, losses in results_dict.items():
        ca = cumavg(losses)
        if len(ca) == 0:
            print(f"  ПРОПУСК (пустой): {name}")
            continue
        ax.plot(ca,
                label=name,
                color=COLORS.get(name, "gray"),
                linestyle=LS.get(name, "-"),
                linewidth=1.8)
    ax.set_yscale("log")
    ax.set_ylim(*ylim)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Итерация (t)", fontsize=10)
    ax.set_ylabel("Накопленная средняя потеря", fontsize=10)
    ax.legend(fontsize=7.5)
    ax.grid(True, alpha=0.3)


fig1, axes = plt.subplots(1, 3, figsize=(20, 6))
fig1.suptitle("California Housing — Huber Loss", fontsize=13)

plot_group(axes[0], res_ca_lin,
           "Линейные признаки x\n(только Coin Betting)",
           ylim=(1e-2, 100))

plot_group(axes[1], res_ca_cb,
           "Фурье-признаки φ(x)\n(только Coin Betting)")

plot_group(axes[2],
           {**res_ca_cb, **res_ca_rv, **res_ca_cm, **res_ca_nn},
           "Фурье-признаки φ(x)\nCoin Betting vs River vs CapyMOA vs MLP")

plt.tight_layout()
plt.savefig("fig_california_all.png", dpi=150, bbox_inches="tight")
plt.show()


fig2, axes2 = plt.subplots(1, 2, figsize=(14, 6))
fig2.suptitle("Diabetes — Фурье-признаки φ(x), Huber Loss", fontsize=13)

plot_group(axes2[0], res_db_cb, "Coin Betting")

plot_group(axes2[1],
           {**res_db_cb, **res_db_rv, **res_db_cm, **res_db_nn},
           "Coin Betting vs River vs CapyMOA vs MLP")

plt.tight_layout()
plt.savefig("fig_diabetes_all.png", dpi=150, bbox_inches="tight")
plt.show()


def print_table(results_dict, title, last_n=200):
    import warnings
    print(f"\n{'=' * 60}")
    print(f"  {title}  (последние {last_n} итераций)")
    print(f"{'=' * 60}")
    print(f"{'Алгоритм':<40} {'Потеря':>10}")
    print("-" * 52)
    for name, losses in results_dict.items():
        v_arr  = np.array(losses[-last_n:], dtype=float)
        finite = v_arr[np.isfinite(v_arr)]
        if len(finite) == 0:
            s = "  diverged"
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                v = np.nanmean(finite)
            s = f"{v:>10.4f}" if np.isfinite(v) else "  diverged"
        print(f"{name:<40} {s}")
    print("=" * 60)


all_ca = {**res_ca_cb, **res_ca_rv, **res_ca_cm, **res_ca_nn}

print("\n=== Длины списков потерь (California Фурье) ===")
for name, losses in all_ca.items():
    print(f"  {name:<40} {len(losses)}")

print_table(res_ca_lin,  "California Housing — Линейные признаки")
print_table(all_ca,      "California Housing — Фурье φ(x), все методы")
print_table(
    {**res_db_cb, **res_db_rv, **res_db_cm, **res_db_nn},
    "Diabetes — Фурье φ(x), все методы",
    last_n=50 
)